<a href="https://colab.research.google.com/github/JManuelTapiaP/Analisis_RappiPlus/blob/main/Analisis_RappiPlus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

Se transformaran datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarnos con la estructura de los datasets del negocio antes de analizarlos.

**Pasos:**

- Importar las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# Explorar datasets
# Explorar orders
print("=== ORDERS ===")
print(orders.shape)
orders.head()

=== ORDERS ===
(25100, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
# Explorar catalog
print("=== CATALOG ===")
print(catalog.shape)
catalog.head()

=== CATALOG ===
(7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
# Explorar marketing
print("=== MARKETING ===")
print(marketing.shape)
marketing.head()

=== MARKETING ===
(1620, 5)


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

---

In [ ]:
#Limpiar orders
print('── ORDERS ──────────────────────────────────────────')

# 1. Convertir fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce') # cambiamos formato
nulos_fecha = orders['fecha_hora_pedido'].isna().sum()
if nulos_fecha > 0: #if para eliminar y llevar registro de cantidad de nulos
    print(f"⚠️  Fechas inválidas: {nulos_fecha} — se eliminan")
    orders = orders.dropna(subset=['fecha_hora_pedido'])
else:
    print("✅ Fechas: sin problemas")

# 2. Nulos en columnas de cálculo → eliminar fila
cols_calculo = ['nombre_producto', 'categoria_producto',
                'cantidad', 'precio_unitario', 'monto_descuento']
nulos_calculo = orders[cols_calculo].isnull().any(axis=1).sum()
if nulos_calculo > 0:
    print(f"⚠️  Nulos en columnas de cálculo: {nulos_calculo} — se eliminan")
    orders = orders.dropna(subset=cols_calculo)
else:
    print("✅ Columnas de cálculo: sin nulos")

# 3. Nulos en columnas de segmentación → rellenar con 'desconocido'
cols_segmentacion = ['pais', 'dispositivo', 'fuente_referencia']
for col in cols_segmentacion:
    nulos = orders[col].isna().sum()
    if nulos > 0:
        print(f"⚠️  Nulos en {col}: {nulos} — se rellenan con 'desconocido'")
        orders[col] = orders[col].fillna('desconocido')
    else:
        print(f"✅ {col}: sin nulos")

# 4. Outliers: cantidades imposibles (> 100)
outliers = (orders['cantidad'] > 100).sum()
if outliers > 0:
    print(f"⚠️  Outliers de cantidad (>100): {outliers} — se eliminan")
    orders = orders[orders['cantidad'] <= 100]
else:
    print("✅ Cantidades: sin outliers")

# 5. Valores numéricos inválidos
neg_cantidad = (orders['cantidad'] <= 0).sum()
neg_precio   = (orders['precio_unitario'] <= 0).sum()
neg_total    = (orders['monto_total'] < 0).sum()
if neg_cantidad > 0 or neg_precio > 0 or neg_total > 0:
    print(f"⚠️  Valores inválidos — cantidad: {neg_cantidad} | precio: {neg_precio} | total: {neg_total} — se eliminan")
    orders = orders[(orders['cantidad'] > 0) &
                    (orders['precio_unitario'] > 0) &
                    (orders['monto_total'] >= 0)]
else:
    print("✅ Valores numéricos: sin problemas")

# 6. Corregir montos inconsistentes
orders['monto_esperado'] = (orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']).round(2) # Recalcular monto_total con la fórmula correcta (formula)
inconsistentes = (abs(orders['monto_total'] - orders['monto_esperado']) > 0.05).sum()
if inconsistentes > 0:
    print(f"⚠️  Montos inconsistentes: {inconsistentes} — se recalculan")
    orders['monto_total'] = orders['monto_esperado'] # Recalcular monto_total con la fórmula correcta
else:
    print("✅ Consistencia de montos: sin problemas")
orders = orders.drop(columns=['monto_esperado'])

# 7. Duplicados por id_pedido
dupes = orders.duplicated(subset='id_pedido').sum()
if dupes > 0:
    print(f"⚠️  Duplicados: {dupes} — se eliminan")
    orders = orders.drop_duplicates(subset='id_pedido', keep='first')
else:
    print("✅ Duplicados: sin problemas")

# 8. Estandarizar capitalización en columnas de texto
cols_texto = ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']
for col in cols_texto:
    orders[col] = orders[col].str.strip().str.title()

print(f"Países únicos: {sorted(orders['pais'].unique())}")
print(f"✅ Texto estandarizado")

print(f"\n→ Orders limpio: {orders.shape}")

── ORDERS ──────────────────────────────────────────
✅ Fechas: sin problemas
⚠️  Nulos en columnas de cálculo: 80 — se eliminan
⚠️  Nulos en pais: 300 — se rellenan con 'desconocido'
⚠️  Nulos en dispositivo: 20 — se rellenan con 'desconocido'
✅ fuente_referencia: sin nulos
⚠️  Outliers de cantidad (>100): 10 — se eliminan
⚠️  Valores inválidos — cantidad: 4 | precio: 0 | total: 4 — se eliminan
✅ Consistencia de montos: sin problemas
⚠️  Duplicados: 100 — se eliminan
Países únicos: ['Argentina', 'Colombia', 'Desconocido', 'Mexico']
✅ Texto estandarizado

→ Orders limpio: (24906, 12)


In [ ]:
# Limpiar catalog
print('── CATALOG ─────────────────────────────────────────')

# 1. Costos inválidos
costos_inv = (catalog['costo_unitario'] <= 0).sum()
if costos_inv > 0:
    print(f"⚠️  Costos inválidos: {costos_inv} — se eliminan")
    catalog = catalog[catalog['costo_unitario'] > 0]
else:
    print("✅ Costos: sin problemas")

# 2. Duplicados por nombre_producto
dupes = catalog.duplicated(subset='nombre_producto').sum()
if dupes > 0:
    print(f"⚠️  Duplicados: {dupes} — se eliminan")
    catalog = catalog.drop_duplicates(subset='nombre_producto', keep='first')
else:
    print("✅ Duplicados: sin problemas")

# 3. Nulos
nulos = catalog.isnull().any(axis=1).sum()
if nulos > 0:
    print(f"⚠️  Nulos: {nulos} — se eliminan")
    catalog = catalog.dropna()
else:
    print("✅ Nulos: sin problemas")

print(f"\n→ Catalog limpio: {catalog.shape}")

── CATALOG ─────────────────────────────────────────
✅ Costos: sin problemas
✅ Duplicados: sin problemas
✅ Nulos: sin problemas

→ Catalog limpio: (7, 4)


In [ ]:
# Limpiar marketing
print('── MARKETING ───────────────────────────────────────')

# 1. Convertir fecha
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
nulos_fecha = marketing['fecha'].isna().sum()
if nulos_fecha > 0:
    print(f"⚠️  Fechas inválidas: {nulos_fecha} — se eliminan")
    marketing = marketing.dropna(subset=['fecha'])
else:
    print("✅ Fechas: sin problemas")

# 2. Estandarizar texto
marketing['pais'] = marketing['pais'].str.strip().str.title()
marketing['id_campaña'] = marketing['id_campaña'].str.strip()
print(f"✅ Texto estandarizado | Países únicos: {sorted(marketing['pais'].unique())}")

# 3. Recuperar canal desde id_campaña cuando es nulo
# Ahora sí usamos pais como referencia, ya estandarizado
paises_conocidos = marketing['pais'].dropna().unique().tolist()

nulos_canal = marketing['canal'].isna().sum()
if nulos_canal > 0:
    print(f"⚠️  Nulos en canal: {nulos_canal} — se recuperan desde id_campaña")
    def extraer_canal(row):
        if pd.isna(row['canal']):
            canal = row['id_campaña']
            for pais in paises_conocidos:
                canal = canal.replace(f'_{pais}', '')
            return canal
        return row['canal']

    marketing['canal'] = marketing.apply(extraer_canal, axis=1)
    print(f"   Nulos restantes: {marketing['canal'].isna().sum()}")
    print(f"   Canales únicos: {sorted(marketing['canal'].unique())}")
else:
    print("✅ Canal: sin nulos")

# 4. Gasto negativo
neg_gasto = (marketing['gasto'] < 0).sum()
if neg_gasto > 0:
    print(f"⚠️  Gasto negativo: {neg_gasto} — se eliminan")
    marketing = marketing[marketing['gasto'] >= 0]
else:
    print("✅ Gasto: sin problemas")

# 5. Duplicados
dupes = marketing.duplicated().sum()
if dupes > 0:
    print(f"⚠️  Duplicados: {dupes} — se eliminan")
    marketing = marketing.drop_duplicates()
else:
    print("✅ Duplicados: sin problemas")

print(f"\n→ Marketing limpio: {marketing.shape}")

── MARKETING ───────────────────────────────────────
✅ Fechas: sin problemas
✅ Texto estandarizado | Países únicos: ['Argentina', 'Colombia', 'Mexico']
⚠️  Nulos en canal: 101 — se recuperan desde id_campaña
   Nulos restantes: 0
   Canales únicos: ['organic', 'paid_search', 'social']
✅ Gasto: sin problemas
✅ Duplicados: sin problemas

→ Marketing limpio: (1620, 5)


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del analisis.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)
print("✅ orders_clean.csv exportado")
print("✅ catalog_clean.csv exportado")
print("✅ marketing_clean.csv exportado")

✅ orders_clean.csv exportado
✅ catalog_clean.csv exportado
✅ marketing_clean.csv exportado


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# Estandarizar categoría antes del join
orders['categoria_producto'] = orders['categoria_producto'].str.normalize('NFC') # estandarizamos ya que algunas categorias estan escritas distintas
catalog['categoria_producto'] = catalog['categoria_producto'].str.normalize('NFC') # estandarizamos ya que algunas categorias estan escritas distintas

# Join orders + catalog para traer costo_unitario
df = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                  on='nombre_producto', how='left')

# Calcular costo total por pedido
df['costo_total'] = df['cantidad'] * df['costo_unitario']

print(f"Tabla base lista: {df.shape}")
print(f"Pedidos sin costo: {df['costo_total'].isna().sum()}")
df.head(5)

Tabla base lista: (24906, 14)
Pedidos sin costo: 0


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,costo_total
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31,378.62
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,25.21
2,order_2,user_3194,2025-05-02,Argentina,Desktop,Social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,176.64,353.28
3,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,25.21
4,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,176.64


In [ ]:
# ── KPIs PRINCIPALES ────────────────────────────────
revenue        = df['monto_total'].sum().round(2)
costo_total    = df['costo_total'].sum().round(2)
gasto_mkt      = marketing['gasto'].sum().round(2)
profit         = (revenue - costo_total - gasto_mkt).round(2)
margen         = (profit / revenue * 100).round(2)

print("=" * 45)
print("  RENTABILIDAD DEL NEGOCIO")
print("=" * 45)
print(f"  Revenue total:        $ {revenue:>15,.2f}")
print(f"  Costo total:          $ {costo_total:>15,.2f}")
print(f"  Gasto marketing:      $ {gasto_mkt:>15,.2f}")
print("  " + "-" * 43)
print(f"  Profit neto:          $ {profit:>15,.2f}")
print(f"  Margen neto:          {margen:>15.2f} %")
print("=" * 45)

if profit > 0:
    print("\n✅ El negocio ES rentable")
else:
    print("\n❌ El negocio NO es rentable")

  RENTABILIDAD DEL NEGOCIO
  Revenue total:        $    9,610,018.94
  Costo total:          $    3,828,869.01
  Gasto marketing:      $    2,871,843.53
  -------------------------------------------
  Profit neto:          $    2,909,306.40
  Margen neto:                    30.27 %

✅ El negocio ES rentable


In [ ]:
# ── TICKET Y CANTIDAD PROMEDIO ───────────────────────
ticket_promedio   = df.groupby('id_pedido')['monto_total'].sum().mean().round(2)
cantidad_promedio = df.groupby('id_pedido')['cantidad'].sum().mean().round(2)

print(f"Ticket promedio por orden:           $ {ticket_promedio:,.2f}")
print(f"Cantidad promedio de productos:        {cantidad_promedio:,.2f}")

# ── PRODUCTO MÁS VENDIDO ─────────────────────────────
top_productos = (df.groupby('nombre_producto')['cantidad']
                   .sum()
                   .sort_values(ascending=False)
                   .reset_index()
                   .rename(columns={'cantidad': 'unidades_vendidas'}))
print(f"\nTop 5 productos más vendidos:")
print(top_productos.head(5).to_string(index=False))

# ── GASTO MARKETING POR CANAL ────────────────────────
gasto_canal = (marketing.groupby('canal')['gasto']
                        .sum()
                        .round(2)
                        .reset_index()
                        .sort_values('gasto', ascending=False))
print(f"\nGasto en marketing por canal:")
print(gasto_canal.to_string(index=False))

Ticket promedio por orden:           $ 385.85
Cantidad promedio de productos:        1.50

Top 5 productos más vendidos:
   nombre_producto  unidades_vendidas
  Vacuum-Pro-Black             6284.0
    Blender-XL-Red             6279.0
   Jacket-Winter-M             6256.0
 Sneakers-Urban-42             6172.0
Laptop-Gaming-16GB             4198.0

Gasto en marketing por canal:
      canal     gasto
     social 976818.37
    organic 972650.96
paid_search 922374.20


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

# Ver estructura completa de events
print(events.columns.tolist()) # Conocer las columnas de eventos para el funnel
print(events.dtypes) # conocer los tipos de datos
print(f"\nTotal filas: {len(events)}")
print(f"\nEventos únicos:")
print(events['nombre_evento'].value_counts())

['id_usuario', 'id_sesion', 'nombre_evento', 'timestamp_evento', 'pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']
id_usuario            object
id_sesion             object
nombre_evento         object
timestamp_evento      object
pais                  object
dispositivo           object
fuente_referencia     object
categoria_producto    object
dtype: object

Total filas: 120000

Eventos únicos:
first_visit         29957
add_to_cart         24157
select_item         23887
begin_checkout      17971
add_payment_info    12018
purchase            12010
Name: nombre_evento, dtype: int64


In [ ]:
# PARTE 1: Totales del funnel
query_totals = '''
WITH cte_first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
cte_select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
cte_begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
cte_add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
)
SELECT
    (SELECT COUNT(*) FROM cte_first_visit)      AS first_visit,
    (SELECT COUNT(*) FROM cte_select_item)      AS select_item,
    (SELECT COUNT(*) FROM cte_add_to_cart)      AS add_to_cart,
    (SELECT COUNT(*) FROM cte_begin_checkout)   AS begin_checkout,
    (SELECT COUNT(*) FROM cte_add_payment_info) AS add_payment_info,
    (SELECT COUNT(*) FROM cte_purchase)         AS purchase
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,first_visit,select_item,add_to_cart,begin_checkout,add_payment_info,purchase
0,7796,7582,7634,7208,6250,6240


In [ ]:
# PARTE 2: Conversión entre pasos
# Definimos lista para nuestro for
etapas = [
    'first_visit',
    'select_item',
    'add_to_cart',
    'begin_checkout',
    'add_payment_info',
    'purchase'
]

# Construir los UNION ALL con for
unions = []
for i, etapa in enumerate(etapas):
    if i == 0:
        # Primera etapa: siempre 100%, sin paso anterior
        unions.append(f"""
SELECT
    '{etapa}' AS etapa,
    {etapa} AS usuarios_unicos,
    100.0 AS pct_vs_primer_paso,
    NULL AS pct_vs_paso_anterior
FROM baseline""")
    else:
        paso_anterior = etapas[i - 1]
        unions.append(f"""
SELECT
    '{etapa}',
    {etapa},
    ROUND({etapa} * 100.0 / first_visit, 2),
    ROUND({etapa} * 100.0 / {paso_anterior}, 2)
FROM baseline""")

# Armar query completa
query_conversion = '''
WITH cte_first_visit AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'first_visit'
),
cte_select_item AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'select_item'
),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_to_cart'
),
cte_begin_checkout AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'begin_checkout'
),
cte_add_payment_info AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'purchase'
),
baseline AS (
    SELECT
        (SELECT COUNT(*) FROM cte_first_visit)      AS first_visit,
        (SELECT COUNT(*) FROM cte_select_item)      AS select_item,
        (SELECT COUNT(*) FROM cte_add_to_cart)      AS add_to_cart,
        (SELECT COUNT(*) FROM cte_begin_checkout)   AS begin_checkout,
        (SELECT COUNT(*) FROM cte_add_payment_info) AS add_payment_info,
        (SELECT COUNT(*) FROM cte_purchase)         AS purchase
)
''' + '\nUNION ALL'.join(unions)

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,etapa,usuarios_unicos,pct_vs_primer_paso,pct_vs_paso_anterior
0,first_visit,7796,100.00,NaN
1,select_item,7582,97.26,97.26
2,add_to_cart,7634,97.92,100.69
3,begin_checkout,7208,92.46,94.42
4,add_payment_info,6250,80.17,86.71
5,purchase,6240,80.04,99.84


## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
LIMIT 5
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
print(user_activity.head())

# Revisión extra para validar datos de user_activity
print()
query_revision = '''
SELECT
    -- Valores únicos de activo
    activo,
    COUNT(*) as total
FROM user_activity
GROUP BY activo

'''
revision = pd.read_sql(query_revision, con=engine)
print(revision)

# Revisar dias_despues_registro
query_dias = '''
SELECT
    MIN(dias_despues_registro) AS minimo,
    MAX(dias_despues_registro) AS maximo,
    COUNT(DISTINCT dias_despues_registro) AS valores_unicos
FROM user_activity
'''
dias = pd.read_sql(query_dias, con=engine)
print(dias)

  id_usuario fecha_actividad  dias_despues_registro  activo
0     user_0      2025-02-05                      7       0
1     user_0      2025-02-12                     14       1
2     user_0      2025-02-19                     21       1
3     user_0      2025-02-26                     28       0
4     user_1      2025-01-14                      7       0

   activo  total
0       0  18685
1       1  13315
   minimo  maximo  valores_unicos
0       7      28               4


In [ ]:
# Retención por cohortes
# ======================

# # Definir las semanas que queremos analizar (7, 14, 21, 28 días)
semanas = [7, 14, 21, 28]

# Generar dinámicamente los CASE WHEN para contar usuarios retenidos
casos_retenidos = []
for i, dias in enumerate(semanas, 1):
    # Cuenta usuarios únicos que estuvieron activos en cada semana específica
    casos_retenidos.append(
        f"COUNT(DISTINCT CASE WHEN a.dias_despues_registro = {dias} AND a.activo = 1 THEN c.id_usuario END) AS retenido_w{i}"
    )

# Generar dinámicamente los retenciones % respecto a los clientes iniciales por semana
casos_porcentajes = []
for i in range(1, len(semanas) + 1):
    casos_porcentajes.append(
        f"ROUND(retenido_w{i} * 100.0 / clientes_iniciales, 2) AS semana_{i}"
    )

# Verificar que tenemos datos antes de ejecutar
print(f"Analizando retención para las semanas: {semanas}")
print(f"Generando {len(casos_retenidos)} métricas de retención...")

Analizando retención para las semanas: [7, 14, 21, 28]
Generando 4 métricas de retención...


In [ ]:
# Armar query completa
query_cohort_retention_final = f'''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM users
),
actividad AS (
    SELECT
        id_usuario,
        dias_despues_registro,
        activo
    FROM user_activity
),
base AS (
    SELECT
        c.mes_cohorte,
        COUNT(DISTINCT c.id_usuario) AS clientes_iniciales,
        {','.join(casos_retenidos)}
    FROM cohortes c
    LEFT JOIN actividad a ON c.id_usuario = a.id_usuario
    GROUP BY c.mes_cohorte
)
SELECT
    mes_cohorte,
    clientes_iniciales,
    {','.join(['retenido_w' + str(i) for i in range(1, len(semanas) + 1)])},
    {','.join(casos_porcentajes)}
FROM base
ORDER BY mes_cohorte
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


# Limpiar formato de mes_cohorte
cohorte_final['mes_cohorte'] = pd.to_datetime(cohorte_final['mes_cohorte']).dt.strftime('%Y-%m')
cohorte_final

,mes_cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,retenido_w4,semana_1,semana_2,semana_3,semana_4
0,2025-01,1627,697,668,656,671,42.84,41.06,40.32,41.24
1,2025-02,1444,611,609,635,575,42.31,42.17,43.98,39.82
2,2025-03,1636,677,705,690,673,41.38,43.09,42.18,41.14
3,2025-04,1606,680,697,663,652,42.34,43.40,41.28,40.60
4,2025-05,1687,695,676,706,679,41.20,40.07,41.85,40.25


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística

   - **H₀ (Hipótesis nula):**  La nueva UI no cambia la tasa de conversión — cualquier diferencia es por azar
   - **H₁ (Hipótesis alternativa):**  La nueva UI sí cambia la tasa de conversión
   
**Test estadístico:** Como 'convirtio' es 0 o 1 (variable binaria), usamos Chi-cuadrado ya que es el test correcto para comparar proporciones entre dos grupos.

**Nivel de significancia alpha:** 0.05 — si el p-value es menor a 0.05, rechazamos H₀.

In [ ]:
import pandas as pd
from scipy import stats

# Cargar dataset del experimento
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

print(experiment.shape)
print(experiment.dtypes)
print(experiment.head())
print(experiment['convirtio'].value_counts())
print(experiment['variante'].value_counts())

(10000, 7)
id_usuario          object
variante            object
convirtio            int64
dispositivo         object
pais                object
duracion_sesion    float64
timestamp           object
dtype: object
   id_usuario     variante  convirtio dispositivo       pais  duracion_sesion  \
0  exp_user_0  tratamiento          0      mobile  Argentina           114.41   
1  exp_user_1  tratamiento          0     desktop     Mexico           170.03   
2  exp_user_2      control          1      mobile   Colombia           140.21   
3  exp_user_3  tratamiento          0      mobile   Colombia           151.45   
4  exp_user_4  tratamiento          0     desktop     Mexico           299.96   

    timestamp  
0  2025-03-28  
1  2025-01-15  
2  2025-03-18  
3  2025-06-03  
4  2025-01-12  
0    8401
1    1599
Name: convirtio, dtype: int64
tratamiento    5035
control        4965
Name: variante, dtype: int64


In [ ]:
# Revisión completa antes de limpiar
print("=== NULOS ===")
print(experiment.isnull().sum())  # Contar nulos por columna

print("\n=== DUPLICADOS ===")
print(f"Duplicados totales: {experiment.duplicated().sum()}")  # Filas completamente iguales
print(f"Duplicados por id_usuario: {experiment.duplicated(subset='id_usuario').sum()}")  # Un usuario no debe aparecer dos veces

print("\n=== VALORES ÚNICOS POR COLUMNA ===")
# Verificar que no haya valores inesperados o mal escritos
print(f"variante:    {experiment['variante'].unique()}")
print(f"convirtio:   {experiment['convirtio'].unique()}")
print(f"dispositivo: {experiment['dispositivo'].unique()}")
print(f"pais:        {experiment['pais'].unique()}")

print("\n=== DURACION SESION ===")
# Verificar que no haya valores negativos o extremos
print(experiment['duracion_sesion'].describe().round(2))

=== NULOS ===
id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64

=== DUPLICADOS ===
Duplicados totales: 0
Duplicados por id_usuario: 0

=== VALORES ÚNICOS POR COLUMNA ===
variante:    ['tratamiento' 'control']
convirtio:   [0 1]
dispositivo: ['mobile' 'desktop']
pais:        ['Argentina' 'Mexico' 'Colombia']

=== DURACION SESION ===
count    10000.00
mean       159.86
std         81.07
min         20.01
25%         89.47
50%        159.72
75%        229.74
max        300.00
Name: duracion_sesion, dtype: float64


In [ ]:
# Convertir timestamp a datetime
experiment['timestamp'] = pd.to_datetime(experiment['timestamp'], errors='coerce')

# Verificar conversion
print(f"Tipo timestamp: {experiment['timestamp'].dtype}")
print("✅ Dataset listo para el test")

Tipo timestamp: datetime64[ns]
✅ Dataset listo para el test


In [ ]:
from scipy import stats

# Separar el dataset en dos grupos según la variante
control     = experiment[experiment['variante'] == 'control']['convirtio']
tratamiento = experiment[experiment['variante'] == 'tratamiento']['convirtio']

# Calcular tasa de conversión de cada grupo
tasa_control     = control.mean().round(4)
tasa_tratamiento = tratamiento.mean().round(4)

print(f"Tasa de conversión control:     {tasa_control * 100:.2f}%")
print(f"Tasa de conversión tratamiento: {tasa_tratamiento * 100:.2f}%")
print(f"Diferencia:                     {(tasa_tratamiento - tasa_control) * 100:.2f}%")

# Tabla de contingencia: cruza variante vs convirtio (0 o 1)
# Es la entrada que necesita el test Chi-cuadrado
tabla = pd.crosstab(experiment['variante'], experiment['convirtio'])
print(f"\nTabla de contingencia:")
print(tabla)

# Aplicar test Chi-cuadrado para comparar proporciones entre dos grupos
chi2, p_value, dof, expected = stats.chi2_contingency(tabla)

print(f"\n=== RESULTADO DEL TEST ===")
print(f"Chi2:    {chi2:.4f}")  # Qué tan diferente es la distribución observada vs esperada
print(f"P-value: {p_value:.4f}")  # Probabilidad de que la diferencia sea por azar
print(f"Alpha:   0.05")  # Umbral de decisión

# Si p_value < alpha rechazamos H₀ y concluimos que la diferencia es real
if p_value < 0.05:
    print("\n✅ Rechazamos H₀ — la diferencia ES estadísticamente significativa")
    print("   La nueva UI sí tiene impacto en la conversión")
else:
    print("\n❌ No rechazamos H₀ — la diferencia NO es estadísticamente significativa")
    print("   No hay evidencia suficiente de que la nueva UI cambie la conversión")

Tasa de conversión control:     15.69%
Tasa de conversión tratamiento: 16.29%
Diferencia:                     0.60%

Tabla de contingencia:
convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820

=== RESULTADO DEL TEST ===
Chi2:    0.6178
P-value: 0.4319
Alpha:   0.05

❌ No rechazamos H₀ — la diferencia NO es estadísticamente significativa
   No hay evidencia suficiente de que la nueva UI cambie la conversión


In [ ]:
# ============================================
# VALIDACIÓN FINAL ANTES DEL DASHBOARD
# ============================================

# 1. Verificar que los 3 CSVs limpios tienen el shape correcto
print("=== SHAPE DE DATASETS LIMPIOS ===")
print(f"Orders:    {orders.shape}")    # Filas y columnas esperadas
print(f"Catalog:   {catalog.shape}")
print(f"Marketing: {marketing.shape}")

# 2. Verificar que todos los productos de orders existen en catalog
productos_orders  = set(orders['nombre_producto'].unique())
productos_catalog = set(catalog['nombre_producto'].unique())
sin_match = productos_orders - productos_catalog
print(f"\n=== CRUCE ORDERS vs CATALOG ===")
print(f"Productos en orders sin match en catalog: {len(sin_match)}")
if len(sin_match) == 0:
    print("✅ Todos los productos tienen costo asignado")
else:
    print(f"⚠️ Productos sin costo: {sin_match}")

# 3. Verificar que los países coinciden entre orders y marketing
paises_orders  = set(orders['pais'].unique()) - {'Desconocido'}
paises_mkt     = set(marketing['pais'].unique())
print(f"\n=== CRUCE PAÍSES ORDERS vs MARKETING ===")
print(f"Países orders:    {sorted(paises_orders)}")
print(f"Países marketing: {sorted(paises_mkt)}")
if paises_orders == paises_mkt:
    print("✅ Países alineados entre datasets")
else:
    print(f"⚠️ Diferencia: {paises_orders.symmetric_difference(paises_mkt)}")

# 4. Verificar rango de fechas consistente
print(f"\n=== RANGO DE FECHAS ===")
print(f"Orders:    {orders['fecha_hora_pedido'].min().date()} → {orders['fecha_hora_pedido'].max().date()}")
print(f"Marketing: {marketing['fecha'].min().date()} → {marketing['fecha'].max().date()}")

# 5. Verificar KPIs clave una última vez
df_val = orders.merge(catalog[['nombre_producto','costo_unitario']], on='nombre_producto', how='left')
df_val['costo_total'] = df_val['cantidad'] * df_val['costo_unitario']

revenue     = df_val['monto_total'].sum().round(2)
costo       = df_val['costo_total'].sum().round(2)
gasto_mkt   = marketing['gasto'].sum().round(2)
profit      = (revenue - costo - gasto_mkt).round(2)

print(f"\n=== KPIs FINALES ===")
print(f"Revenue:         $ {revenue:>15,.2f}")
print(f"Costo:           $ {costo:>15,.2f}")
print(f"Gasto marketing: $ {gasto_mkt:>15,.2f}")
print(f"Profit:          $ {profit:>15,.2f}")
print(f"Margen:            {(profit/revenue*100):.2f}%")

=== SHAPE DE DATASETS LIMPIOS ===
Orders:    (24906, 12)
Catalog:   (7, 4)
Marketing: (1620, 5)

=== CRUCE ORDERS vs CATALOG ===
Productos en orders sin match en catalog: 0
✅ Todos los productos tienen costo asignado

=== CRUCE PAÍSES ORDERS vs MARKETING ===
Países orders:    ['Argentina', 'Colombia', 'Mexico']
Países marketing: ['Argentina', 'Colombia', 'Mexico']
✅ Países alineados entre datasets

=== RANGO DE FECHAS ===
Orders:    2025-01-01 → 2025-06-30
Marketing: 2025-01-01 → 2025-06-29

=== KPIs FINALES ===
Revenue:         $    9,610,018.94
Costo:           $    3,828,869.01
Gasto marketing: $    2,871,843.53
Profit:          $    2,909,306.40
Margen:            30.27%


In [ ]:
# Exportar funnel y cohortes para Power BI (extra)
conversion.to_csv('funnel_conversion.csv', index=False)
cohorte_final.to_csv('cohortes_retencion.csv', index=False)

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---